# Quant Backtester - Moving Average Crossover Research

This notebook demonstrates the research capabilities of the `quant-backtester` engine. 
We will:
1. Setup the environment and generate synthetic data.
2. Run a parameter sweep for the `MovingAverageCrossStrategy`.
3. Analyze and visualize the results (Heatmaps, Equity Curves, Drawdowns).

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Ensure project root is in path to import src
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.strategy import MovingAverageCrossStrategy
from src.research import StrategyResearchRunner
from src.visualization import plot_equity_curve, plot_drawdown, plot_param_heatmap
from src.data_handler import DataHandler
from src.portfolio import Portfolio
from src.engine import BacktestEngine
from src.performance import create_equity_curve, calculate_drawdown

## 1. Data Setup
We will generate a synthetic price series with some trend and noise to verify the strategy logic.

In [ ]:
def generate_synthetic_data(file_path, length=500):
    np.random.seed(42)
    dates = [datetime(2023, 1, 1) + timedelta(days=i) for i in range(length)]
    
    # Random Walk with drift
    returns = np.random.normal(0.0005, 0.02, length)
    price = 100 * np.cumprod(1 + returns)
    
    df = pd.DataFrame({'Date': dates, 'Close': price, 'Volume': 1000})
    df.to_csv(file_path, index=False)
    print(f"Generated data at {file_path}")
    return df

data_path = os.path.join(project_root, 'data', 'synthetic_research.csv')
os.makedirs(os.path.dirname(data_path), exist_ok=True)
generate_synthetic_data(data_path, length=500).head()

## 2. Parameter Sweep
We will test a grid of `short_window` and `long_window` parameters to find the optimal combination based on Sharpe Ratio.

In [ ]:
param_grid = {
    "short_window": [5, 10, 20, 30],
    "long_window": [40, 50, 60, 80, 100, 120]
}

runner = StrategyResearchRunner(
    data_path=data_path,
    symbol="SYNTH",
    strategy_cls=MovingAverageCrossStrategy,
    param_grid=param_grid
)

results_df = runner.run()

In [ ]:
# Display Top 5 Strategies by Sharpe Ratio
top_strategies = results_df.sort_values(by='Sharpe Ratio', ascending=False).head(5)
top_strategies

## 3. Visualization
### Parameter Heatmap
Visualizing the stability of the parameter space.

In [ ]:
plot_param_heatmap(
    results_df, 
    x_param='short_window', 
    y_param='long_window', 
    metric='Sharpe Ratio'
)

### Best Strategy Deep Dive
Let's re-run the best performing strategy to visualize its detailed Equity Curve and Drawdown.

In [ ]:
best_params = top_strategies.iloc[0]
short_w = int(best_params['short_window'])
long_w = int(best_params['long_window'])

print(f"Running best strategy: Short={short_w}, Long={long_w}")

# 1. Setup
portfolio = Portfolio()
data_handler = DataHandler(data_path, "SYNTH")
strategy = MovingAverageCrossStrategy(short_window=short_w, long_window=long_w)

# 2. Run
engine = BacktestEngine(data_handler, strategy, portfolio)
engine.run()

# 3. Extract Metrics
equity_curve = create_equity_curve(portfolio.history)
drawdown = calculate_drawdown(equity_curve)

# 4. Plot
plot_equity_curve(equity_curve, title=f"Best Strategy (Short={short_w}, Long={long_w})")
plot_drawdown(drawdown, title="Strategy Drawdown")